In [1]:
from sympy import *
import copy
import time
import pickle

# Classes

In [3]:
# If this class is modified, the pickling of ad matrices should be re-executed

class T_symb_basis_elt:
    def __init__(self,str_rep,wght):
        self.str_rep=str_rep
        
        self.vec_rep=[0]*(2*m+5)
        self.vec_rep[['Y','H','E','X','e1','e2','e3','e4','e5','e6','N'].index(str_rep)]=1
        
        self.wght=wght
        self.ad_matrix='UNKNOWN'
        self.cochain_ad_matrices=['UNKNOWN','UNKNOWN','UNKNOWN','UNKNOWN']
        self.ext_ad_matrices=['UNKNOWN','UNKNOWN','UNKNOWN','UNKNOWN']
        
    def __str__(self):
        return self.str_rep
    
    def __repr__(self):
        return self.str_rep
    
    def __lt__(self,other):
        return T_symb_basis.index(self)<T_symb_basis.index(other)
    
    def __gt__(self,other):
        return T_symb_basis.index(self)>T_symb_basis.index(other)
    
    def __le__(self,other):
        return T_symb_basis.index(self)<=T_symb_basis.index(other)

    def __ge__(self,other):
        return T_symb_basis.index(self)>=T_symb_basis.index(other)
    
    def __add__(self,other):
        if type(other)==type(self):
            result=[0]*len(T_symb_basis)
            result[T_symb_basis.index(self)]+=1
            result[T_symb_basis.index(other)]+=1
            return T_symb_elt(result)
        result=other.vec_rep
        result[T_symb_basis.index(self)]+=1
        return T_symb_elt(result)
    
    def __neg__(self):
        result=[0]*len(T_symb_basis)
        result[T_symb_basis.index(self)]=-1
        return T_symb_elt(result)
    
    def __sub__(self,other):
        if type(other)==type(self):
            return self+(-other)
        result=[-A for A in other.vec_rep]
        result[T_symb_basis.index(self)]+=1
        return T_symb_elt(result)
    
    def __mul__(self,other):
        result=[0]*len(T_symb_basis)
        result[T_symb_basis.index(self)]=other
        return T_symb_elt(result)
    
    def __rmul__(self,other):
        return(self*other)

In [4]:
class T_symb_elt:
    
    def __init__(self,vec_rep):
        '''vec_rep: a list of length len(T_symb_basis) with integer entries'''
        self.vec_rep=vec_rep
    
    def __str__(self):
        if self.vec_rep==[0]*len(T_symb_basis):
            return '0'
        
        result=''
        cntr=0
        while result=='':
            if self.vec_rep[cntr]!=0:
                if self.vec_rep[cntr]==1:
                    result=str(T_symb_basis[cntr])
                elif self.vec_rep[cntr]==-1:
                    result='-'+str(T_symb_basis[cntr])
                else:
                    result = str(self.vec_rep[cntr])+'*'+str(T_symb_basis[cntr])
            cntr+=1
        for i in range(cntr,len(T_symb_basis)):
            if self.vec_rep[i]==1:
                result+=' + '+str(T_symb_basis[i])
            elif self.vec_rep[i]==-1:
                result+=' - '+str(T_symb_basis[i])
            elif self.vec_rep[i]!=0:
                result+=' + '+str(self.vec_rep[i])+'*'+str(T_symb_basis[i])
        return result
    
    def __eq__(self,other):
        if other==0:
            return self.vec_rep==[0]*(len(T_symb_basis))
        return self.vec_rep==other.vec_rep
    
    def __repr__(self):
        return str(self)
    
    def __neg__(self):
        return(T_symb_elt([-A for A in self.vec_rep]))
    
    def __add__(self,other):
        if other==0: return copy.copy(self)
        if type(other)==type(self):
            return T_symb_elt([self.vec_rep[i]+other.vec_rep[i] for i in range(len(T_symb_basis))])
        return other+self
    
    def __sub__(self,other):
        return self+(-other)
            
    def __mul__(self,other):
        return T_symb_elt([other*A for A in self.vec_rep])
    
    def __rmul__(self,other):
        return self*other

In [5]:
class cochain:
    
    def __init__(self,coeff_dict={},wght='UNKNOWN',deg='UNKNOWN',is_in_norm_space='UNKNOWN'):
        '''coeff_dict:  tuple of T_symb_basis_elts objs as keys, coeffs as values
           wght (optional): homog. wght, if known'''
        self.deg=deg   ## The zero cochain has wght and deg 'Nil'
        self.wght=wght 
        self.coeff_dict=remove_zeros({sort_cochain_tuple(A):coeff_dict[A] for A in coeff_dict})
        self.is_in_norm_space=is_in_norm_space
        self.matrix_rep='UNKNOWN'
        set_matrix_rep(self)
                
                
    def __eq__(self,other):
        if other==0:
            return self.coeff_dict=={}
        return self.coeff_dict==other.coeff_dict
        
    def __add__(self,other):
        # zero cochain case
        if other==0: return copy.copy(self)
        if self.deg=='Nil': return copy.copy(other)
        if other.deg=='Nil': return copy.copy(self)
        
        result=copy.copy(self)
        
        ## If one wght is UNKNOWN
        if self.wght==other.wght or other.wght=='UNKNOWN':
            result.wght=self.wght
        elif self.wght=='UNKNOWN':
            result.wght=other.wght
        else:
            result.wght='UNKNOWN'
            
        ## If one deg is UNKNOWN, or degrees don't match:
        if self.deg=='UNKNOWN' or other.deg=='UNKNOWN' or self.deg!=other.deg:
            result.matrix_rep='UNKNOWN'
        else: result.matrix_rep=self.matrix_rep+other.matrix_rep
        
        
        result.coeff_dict=merge_coeff_dicts(self.coeff_dict,other.coeff_dict)
        if self.is_in_norm_space=='UNKNOWN' or other.is_in_norm_space=='UNKNOWN':
            result.is_in_norm_space='UNKNOWN'
        else: 
            result.is_in_norm_space=self.is_in_norm_space and other.is_in_norm_space
            
        return result
    
    def __neg__(self):
        return cochain({A:-self.coeff_dict[A] for A in self.coeff_dict},
                       wght=self.wght,deg=self.deg,is_in_norm_space=self.is_in_norm_space)
        
    def __sub__(self,other):
        return self+(-neg_other)
    
    def __mul__(self,k):
        kself=copy.copy(self)
        kself.coeff_dict={A:k*self.coeff_dict[A] for A in self.coeff_dict}
        kself.matrix_rep=k*self.matrix_rep
        return kself
    
    def __rmul__(self,k):
        return self*k
    
    def coboundary(self):
        result=cochain()
        for key in self.coeff_dict():
            result=result+self.coeff_dict[key]*coboundary_dict[key]
        return result
    
    def display_coeff_dict(self):
        print({B: self.coeff_dict[B] for B in self.coeff_dict})
    
    def __str__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __repr__(self):
        return str_from_coeff_dict(self.coeff_dict)

In [6]:
def remove_antisymm_zeros(coeff_dict):
    '''coeff_dict: a coeff_dict for an exterior vector
       result: coeff_dict, but with keys like (e1,e2,e1) removed'''
    result_dict=copy.copy(coeff_dict)
    for key in list(result_dict):
        if len(set(key))!=len(key):
            result_dict.pop(key)
    return result_dict

In [7]:
class ext_T_symb_elt():
    
    def __init__(self,coeff_dict={},wght='UNKNOWN',deg='UNKNOWN'):
        
        ## I think these lines are unecessary; at least deg
        #  is set in set_vec_rep
        if coeff_dict=={} or list(coeff_dict.keys())==[()]: ## The zero wedge has wght and deg 'Nil'
            self.deg='Nil'
            self.wght='Nil'
        else:
            self.deg=deg   
            self.wght=wght 
        
        self.coeff_dict=remove_antisymm_zeros(remove_zeros({sort_basis_tuple(A)[0]:
                                      sort_basis_tuple(A)[1]*coeff_dict[A] for A in coeff_dict}))
        self.vec_rep=None
        set_vec_rep(self)
    
    def __str__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __repr__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __eq__(self,other):
        if other==0:
            return self.coeff_dict=={}
        return self.coeff_dict==other.coeff_dict
    
    def __neg__(self):
        return ext_T_symb_elt({A:-self.coeff_dict[A] for A in self.coeff_dict},wght=self.wght,deg=self.deg)
    
    def __add__(self,other):
        new_wght='UNKNOWN'
        if self.wght!='UNKNOWN' and other.wght!='UNKNOWN':
            new_wght=self.wght+other.wght
        new_deg='UNKNOWN'
        if self.deg!='UNKNOWN' and other.deg!='UNKNOWN':
            new_deg=self.deg+other.deg
        return(ext_T_symb_elt(merge_coeff_dicts(self.coeff_dict, other.coeff_dict),wght=new_wght,deg=new_deg))
    
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,other):
        return(ext_T_symb_elt({A:other*self.coeff_dict[A]},wght=self.wght,deg=self.deg))
    
    def __rmul(self,other):
        return self*other
    
    def wedge(self,other):
        '''other: an ext_T_symb_elt or cochain object
           returns: the wedge product of self and other as an ext_T_symb_elt
           Note: Functionality only for wedges of deg <4, since vec_rep is used'''
        
        if type(other)==type(ext_T_symb_elt({})):
            result=ext_T_symb_elt({})
            for A in self.coeff_dict:
                for B in other.coeff_dict:
                    new_wedge=wedge_tuples(A,B)
                    if new_wedge!='Nil':
                        result+=ext_T_symb_elt({new_wedge[0]:new_wedge[1]*self.coeff_dict[A]*other.coeff_dict[B]})
            return result
        
        if type(other)==type(cochain({})):
            result=cochain({})
            for A in self.coeff_dict:
                for B in other.coeff_dict:
                    new_wedge=wedge_tuples(A,B[0:len(B)-1])
                    if new_wedge!='Nil':
                        new_cochain=(new_wedge[0]+B[len(B)-1:len(B)],new_wedge[1])
                        result+=cochain({new_cochain[0]:new_cochain[1]*self.coeff_dict[A]*other.coeff_dict[B]})
            return result
        
        if type(other)==type(T_symb_elt([0]*len(T_symb_basis))):
            result=cochain({})
            for ext_tuple in self.coeff_dict:
                for i in range(len(T_symb_basis)):
                    A=T_symb_basis[i]
                    coeff=self.coeff_dict[ext_tuple]*other.vec_rep[i]
                    result+=cochain({ext_tuple+(A,):coeff})
            return result
        
        if type(other)==type(X):
            vec=[0]*len(T_symb_basis)
            vec[T_symb_basis.index(other)]=1
            return self.wedge(T_symb_elt(vec))
                    

# Functions Utilized in Classes

In [8]:
def wedge_tuples(tuple1,tuple2):
    '''tuple1,tuple2: tuples of T_symb_basis_elt objects
       returns: (wedge,sgn), where wedge is a tuple representing tuple1 wedge tuple2
               and sgn is -1 or 1, or'Nil' if the wedge product is zero '''
    if len(tuple1)==0 or len(tuple2)==0:
        return 'Nil'
    #check for repeats
    if len(set(tuple1).union(set(tuple2)))!=len(tuple1)+len(tuple2):
        return 'Nil'
    return sort_basis_tuple(tuple1+tuple2)

In [9]:
def ext_tuple_wght(rep):
    '''rep: a tuple representing an ext_T_symb_elt
       returns: the wght of the corresponding exterior element'''
    return sum([A.wght for A in rep])

def cochain_tuple_wght(rep):
    '''rep: a tuple representing a cochain
       returns: the wght of the corresponding cochain'''
    return -sum([A.wght for A in rep[0:len(rep)-1]])+rep[len(rep)-1].wght

In [10]:
## Note: It's important for these methods that 
#  we keep zero entries out of coeff_dicts
def cochain_deg(c):
    '''c: a cochain object
       returns: deg(c) if c has homogeneous deg, 
         'Nil' if c is the zero cochain,'UNKNOWN' otherwise'''
    if len(c.coeff_dict.keys())==0:
        return 'Nil' # The zero cochain
    deg=len(list(c.coeff_dict.keys())[0])-1
    for A in c.coeff_dict:
        if len(A)-1!=deg:
            return 'UNKNOWN'
    return deg
    
def ext_T_symb_elt_deg(ext_elt):
    '''ext_elt: an ext_T_symb_elt object
       returns: deg(ext_elt) if ext_elt has homogeneneous deg,
         'Nil' if ext_elt is the zero wedge, 'UNKNOWN' otherwise'''
    if len(ext_elt.coeff_dict.keys())==0:
        return 'Nil'
    deg=len(list(ext_elt.coeff_dict.keys())[0])
    for A in ext_elt.coeff_dict:
        if len(A)!=deg:
            return 'UNKNOWN'
    return deg

In [11]:
def cochain_wght(c):
    '''c: a cochain object
       returns: wght(c) if c has homogeneous wght, 'UNKNOWN' otherwise'''
    if len(c.coeff_dict.keys())==0:
        return 'Nil' # The zero cochain
    wght=cochain_tuple_wght(list(c.coeff_dict.keys())[0])
    for A in c.coeff_dict:
        if cochain_tuple_wght(A)!=wght:
            return 'UNKNOWN'
    return wght 

def ext_T_symb_elt_wght(ext_elt):
    '''ext_elt: an ext_T_symb_elt object
       returns: wght(ext_elt) if ext_elt has homogeneous wght, 'UNKNOWN' otherwise'''
    if len(ext_elt.coeff_dict.keys())==0:
        return 'Nil' # The zero cochain
    wght=ext_tuple_wght(list(c.coeff_dict.keys())[0])
    for A in c.coeff_dict:
        if ext_tuple_wght(A)!=wght:
            return 'UNKNOWN'
    return wght 

In [12]:
## Computing and setting attributes of a cochain
def cochain_in_norm_space(c):
    '''c: a cochain object
       returns: True if c is a positive cochain in 
            Hom(Wedge(g_-),g), False otherwise'''
    for key in c.coeff_dict:
        if cochain_tuple_wght(key)<=0:
            return False
        for i in range(len(key)-1):
            if key[i].wght>=0:
                return False
    return True

In [13]:
def set_matrix_rep(c):
    '''c: a cochain obj
       returns: None
       Sets c.matrix_rep to a matrix if c has homogeneous degree,
       'UNKNOWN' if not, and 'Nil' if c is the zero cochain'''
    c.deg=cochain_deg(c)
    if c.deg=='UNKNOWN':
        c.matrix_rep='UNKNOWN'
    elif c.deg=='Nil':
        c.matrix_rep='Nil'
    else:
        c.matrix_rep=zeros(len(T_symb_basis),binomial(len(T_symb_basis),c.deg))
        for A in c.coeff_dict:
            if c.deg!=0:
                dom_index=ext_basis[c.deg].index(A[0:len(A)-1])
                codom_index=T_symb_basis.index(A[len(A)-1])
                c.matrix_rep[codom_index,dom_index]+=c.coeff_dict[A]
            if c.deg==0:
                codom_index=T_symb_basis.index(A[len(A)-1])
                c.matrix_rep[codom_index,0]=c.coeff_dict[A]

In [14]:
    def set_vec_rep(ext_elt):
        '''ext_elt: an ext_T_symb_elt object
           returns: None
           Sets ext_elt.vec_rep to a representative vector (1 x ext_elt.deg) matrix'''
        ext_elt.deg=ext_T_symb_elt_deg(ext_elt)
        if ext_elt.deg=='UNKNOWN':
            ext_elt.vec_rep='UNKNOWN'
            return None
        if ext_elt.deg=='Nil':
            ext_elt.vec_rep='Nil'
            return None
        result=zeros(len(ext_basis[ext_elt.deg]),1)
        for A in ext_elt.coeff_dict:
            i=ext_basis[ext_elt.deg].index(A)
            result[i,0]=ext_elt.coeff_dict[A]
        ext_elt.vec_rep=result

In [15]:
def str_from_coeff_dict(coeff_dict):
    '''coeff_dict: a dict with printable keys and integer values
       returns: a string representing the dict'''
    no_zeros=remove_zeros(coeff_dict)
    if coeff_dict=={}:
            return '0'
    key_list=list(coeff_dict.keys())
    result=''

    for key in key_list:
        if coeff_dict[key]==1:
            result+=' + '+str(key)
        elif coeff_dict[key]==-1:
            result+=' - '+str(key)
        elif coeff_dict[key]!=0:
            result+=' + '+str(coeff_dict[key])+'*'+str(key)
    if result[0:3]==' + ':
        return result[3:len(result)]
    return result[1:len(result)]

In [16]:
def remove_zeros(coeff_dict):
    '''coeff_dict: a dict with integer values
       returns: a copy of coeff_dict with all keys of value 0 removed'''
    return{A:coeff_dict[A] for A in coeff_dict if coeff_dict[A]!=0 and A!=()}

In [17]:
## For addition of cochains
def merge_coeff_dicts(dict1,dict2):
    result={}
    for key in set(dict1.keys()).union(set(dict2.keys())):
        coeff=0
        if key in dict1:
            coeff+=dict1[key]
        if key in dict2:
            coeff+=dict2[key]
        result[key]=coeff
    return remove_zeros(result)

In [18]:
def permutation_sign(it_1,it_2):
    '''tuple_1, tuple_2: iterables containing the same elements
       returns: the sign of the permutation taking it_1 to it_2'''
    cnt=0
    for i in range(len(it_1)):
        for j in range(i+1,len(it_1)):
            if it_2.index(it_1[j])<it_2.index(it_1[i]):
                cnt+=1
    return (-1)**cnt

def sort_basis_tuple(basis_tuple):
    '''basis_tuple: a tuple of T_symb_basis_elt objs
       returns: a tuple containing an rearrangement of basis_tuple of descending degree, 
       and the sign of the permutation (either -1 or 1)'''
    basis_list=list(basis_tuple)
    sorted_list=basis_list.copy()
    sorted_list.sort(key=lambda A:T_symb_basis.index(A))
    return(tuple(sorted_list),permutation_sign(basis_list,sorted_list))

In [19]:
def sort_cochain_tuple(cochain_tuple):
    '''cochain_tuple: a tuple of T_symb_basis_elt objs, representing a cochain
       returns: a rearrangement of cochain_tuple, descending in degree,
                but leaving the final element of cochain_tuple invariant'''
    
    ext_list=list(cochain_tuple[0:len(cochain_tuple)-1])
    ext_list.sort(key=lambda A:T_symb_basis.index(A))
    return(tuple(ext_list+[cochain_tuple[len(cochain_tuple)-1]]))

In [20]:
def apply_cochain_map(c,ext_elt):
    '''c: a cochain object
       ext_elt: an ext_T_symb_elt object
       returns: T_symb_elt object representing c(ext_elt) or None if c.deg!=ext_elt.deg'''
    
    c.deg=cochain_deg(c)
    ext_elt.deg=ext_T_symb_elt_deg(ext_elt)
    
    if c.deg!=ext_elt.deg:
        return None
    c.matrix_rep
    ## to do!!

In [21]:
def ext_T_symb_elt_from_vec(vec,deg):
    '''vec: a column vector of length len(ext_basis[deg])
       deg: the deg of the ext elt to be represented
       returns: an ext_T_symb_elt representing the vector'''
    return ext_T_symb_elt({ext_basis[deg][i]:vec[i,0] for i in range(len(ext_basis[deg]))})

In [22]:
def cochain_from_vec(vec,deg):
    '''vec: a column vector of length len(ext_basis[deg])
       deg: the deg of the cochain elt to be represented
       returns: a cochain represented by vec'''
    return cochain({cochain_basis_tuples[deg][i]:vec[i,0] for i in range(len(cochain_basis[deg]))})